# Tutorial 04 — Post-training I: prompting, instruction tuning and LoRA

[Lecture notes](../lectures/04_adaptation.md) · [Slides](../slides/04_adaptation.pptx) · [Exercise sheet](../exercises/04_adaptation.md)

**40 minutes, CPU, NumPy only.** You will fit a low-rank adapter to a frozen linear model, mark which tokens carry loss in response-only fine-tuning, and watch a toy in-context classifier change its answers when the same four demonstrations are reordered. Everything here is synthetic. Nothing measures a real language model.

| Minutes | Step |
|---|---|
| 0–5 | Build the adaptation task |
| 5–15 | Train the adapter and check the frozen weights |
| 15–20 | Response-only labels |
| 20–30 | Demonstration order in a toy prompt |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import itertools
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. An adaptation task

The base model is a frozen 12×8 matrix `Wbase`. The targets come from `Wbase` plus a rank-2 change. The adapter is `A` (12×2, random) times `B` (2×8, zero). Inputs are row vectors, so the model is `X @ (Wbase + scale * A @ B)`.

Predict the two parameter counts before you run the cell.

In [2]:
n,din,dout,r=128,12,8,2
X=rng.normal(size=(n,din)); Xtest=rng.normal(size=(64,din))
Wbase=rng.normal(0,.2,(din,dout))
true_delta=rng.normal(0,.2,(din,r))@rng.normal(0,.2,(r,dout))
Y=X@(Wbase+true_delta); Ytest=Xtest@(Wbase+true_delta)
A=rng.normal(0,.1,(din,r)); B=np.zeros((r,dout)); scale=1.0
frozen=Wbase.copy()
def mse(x,y): return np.mean((x@(Wbase+scale*A@B)-y)**2)
print("base test MSE",mse(Xtest,Ytest),"adapter/full parameters",r*(din+dout),din*dout)

base test MSE 0.05412808238153114 adapter/full parameters 40 96


The adapter has 40 numbers against 96 in the full matrix. `B` is zero, so the printed error, 0.054, is the base model's own. What would the adapter count be at rank 4, and when does a rank-$r$ adapter stop being smaller than the matrix?

## 2. Train the adapter only

Both gradients are computed before either factor is updated. The cell asserts that the gradient of `A` is zero at the first step and that `Wbase` never changes.

In [3]:
initial=mse(Xtest,Ytest)
for step in range(401):
    err=X@(Wbase+scale*A@B)-Y
    G=2*X.T@err/err.size
    dA=scale*G@B.T; dB=scale*A.T@G
    if step==0: assert np.allclose(dA,0) and np.linalg.norm(dB)>0
    A-=.5*dA; B-=.5*dB
    if step%100==0: print(step,"train",mse(X,Y),"test",mse(Xtest,Ytest))
assert np.array_equal(Wbase,frozen)
assert mse(Xtest,Ytest)<initial

0 train 0.06146098485189066 test 0.05386557729664078
100 train 0.0026530153745442002 test 0.004125273080141428
200 train 9.402783747706043e-06 test 1.9301633920292944e-05
300 train 5.799000950589515e-08 test 1.3488593607492863e-07
400 train 5.892575182561166e-10 test 1.3255859644546835e-09


Test error falls from 0.054 to about $10^{-9}$. The fit is this good because the true change has rank 2 and the adapter has rank 2. A rank-1 adapter could not do this. Which part of the true change would it leave unfitted?

## 3. Response-only labels

Token IDs 10, 11, 12 are the prompt, 20 and 21 are the response, 2 is the end-of-sequence token and 0 is padding. The label −100 tells the loss to skip a position. The model still reads every prompt token.

In [4]:
# IDs 10/11/12 are an illustrative prompt; 20/21 are response; 2 is EOS; 0 is padding.
input_ids=np.array([[10,11,12,20,21,2,0,0]])
attention_mask=np.array([[1,1,1,1,1,1,0,0]])
labels=input_ids.copy(); labels[:,:3]=-100; labels[attention_mask==0]=-100
print("inputs",input_ids,"labels",labels)
assert np.array_equal(labels,np.array([[-100,-100,-100,20,21,2,-100,-100]]))
assert (input_ids[:,:3]!=-100).all()

inputs [[10 11 12 20 21  2  0  0]] labels [[-100 -100 -100   20   21    2 -100 -100]]


Three of eight positions carry loss: the two response tokens and the end token. If the label for token 2 were also −100, what would the tuned model never learn to do?

## 4. Demonstration order in a toy prompt

A few-shot prompt holds $k$ labelled demonstrations and then a query. Lu et al. (2021) found that reordering the same demonstrations can move a model from near its best accuracy to near chance, and Zhao et al. (2021) found that models lean towards the labels nearest the end of the prompt.

This cell builds a toy with that one property. Each example is a 6-number feature vector with a label of +1 or −1. The "model" compares the query with each of the four demonstrations and takes a weighted vote. The demonstration in position $i$ gets weight $\gamma^{k-1-i}$, so with $\gamma<1$ the last one counts most. With $\gamma=1$ every position counts the same.

Predict the accuracy range over the 24 orderings for $\gamma=1$ and for $\gamma=0.5$.

In [5]:
k, d_feat = 4, 6
mu = np.zeros(d_feat); mu[0] = 1.0                      # class +1 sits at +mu, class -1 at -mu

def draw(n):
    y = rng.choice([-1, 1], size=n)
    return y[:, None] * mu + rng.normal(0, 1.0, (n, d_feat)), y

demo_y = np.array([1, 1, -1, -1])                        # two demonstrations per class
demo_x = demo_y[:, None] * mu + rng.normal(0, 1.0, (k, d_feat))
dev_x, dev_y = draw(40); test_x, test_y = draw(2000)

def icl_predict(queries, order, gamma):
    """Similarity-weighted vote over the demonstrations in `order`; later positions weigh more when gamma < 1."""
    xs, ys = demo_x[list(order)], demo_y[list(order)]
    w = gamma ** np.arange(k - 1, -1, -1)                # the last demonstration has weight 1
    sim = np.exp(-((queries[:, None, :] - xs[None]) ** 2).sum(-1) / (2 * d_feat))
    return np.where((sim * w * ys).sum(1) >= 0, 1, -1)

orders = list(itertools.permutations(range(k)))
for gamma in (1.0, 0.5):
    acc = np.array([(icl_predict(test_x, o, gamma) == test_y).mean() for o in orders])
    said_pos = np.array([(icl_predict(test_x, o, gamma) == 1).mean() for o in orders])
    print(f"gamma {gamma}: test accuracy over 24 orders  min {acc.min():.3f}  mean {acc.mean():.3f}  max {acc.max():.3f}"
          f"   share answered +1: {said_pos.min():.3f} to {said_pos.max():.3f}")
    if gamma == 1.0: assert acc.max() - acc.min() < 1e-12       # no recency weighting, no order effect
    else: assert acc.max() - acc.min() > 0.1                     # recency weighting makes order matter

gamma 1.0: test accuracy over 24 orders  min 0.714  mean 0.714  max 0.714   share answered +1: 0.710 to 0.710
gamma 0.5: test accuracy over 24 orders  min 0.483  mean 0.584  max 0.795   share answered +1: 0.002 to 1.000


With $\gamma=1$ all 24 orderings score 0.714. With $\gamma=0.5$ the same four demonstrations score between 0.483 and 0.795, and some orderings answer +1 for every query. Look at the last column: which label sits at the end of the prompt in the orderings that always answer +1?

### The prompt is a hyperparameter

Choose the ordering on 40 development queries, then score that one ordering on the 2,000 test queries.

In [6]:
dev_acc = np.array([(icl_predict(dev_x, o, 0.5) == dev_y).mean() for o in orders])
test_acc = np.array([(icl_predict(test_x, o, 0.5) == test_y).mean() for o in orders])
best, worst = dev_acc.argmax(), test_acc.argmin()
print("order chosen on dev:", orders[best], "labels in that order:", demo_y[list(orders[best])])
print(f"its dev accuracy {dev_acc[best]:.3f}   its test accuracy {test_acc[best]:.3f}   mean test accuracy over orders {test_acc.mean():.3f}")
print("worst order on test:", orders[worst], "labels in that order:", demo_y[list(orders[worst])], f"accuracy {test_acc[worst]:.3f}")
assert test_acc[best] > test_acc.mean()

order chosen on dev: (1, 3, 0, 2) labels in that order: [ 1 -1  1 -1]
its dev accuracy 0.825   its test accuracy 0.795   mean test accuracy over orders 0.584
worst order on test: (2, 3, 0, 1) labels in that order: [-1 -1  1  1] accuracy 0.483


The chosen ordering alternates the labels and scores 0.825 on development and 0.795 on test. The development score is the higher of the two because it is the best of 24 noisy scores on 40 queries. Report the test number, and report how many prompts you tried. The worst ordering puts both +1 demonstrations last.

## 5. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Response-only labels for a batch.** Rows have prompts of different lengths and different amounts of padding. Return a copy of `input_ids` with −100 at every prompt position and every padding position.

In [ ]:
# YOUR TURN
def response_only_labels(input_ids, attention_mask, prompt_lens):
    """Copy of input_ids with -100 on the first prompt_lens[i] positions of row i and wherever attention_mask is 0."""
    raise NotImplementedError("your turn")

In [8]:
ids = np.array([[10, 11, 12, 20, 21, 2, 0, 0],
                [10, 11, 20, 21, 22, 23, 2, 0],
                [10, 11, 12, 13, 14, 20, 2, 0]])
mask = (ids != 0).astype(int)
out = response_only_labels(ids, mask, [3, 2, 5])
assert np.array_equal(out, np.array([[-100, -100, -100, 20, 21, 2, -100, -100],
                                     [-100, -100, 20, 21, 22, 23, 2, -100],
                                     [-100, -100, -100, -100, -100, 20, 2, -100]]))
assert ids[0, 0] == 10                                   # the inputs are not changed
print("supervised positions per row:", (out != -100).sum(1))
print("A passed")

supervised positions per row: [3 5 2]
A passed


**B. Count and merge.** Write the adapter's parameter count, and merge a trained adapter into the base matrix so that inference needs one matrix. Use the course convention $W' = W + sAB$.

In [ ]:
# YOUR TURN
def lora_param_count(d_in, d_out, r):
    """Number of trainable numbers in a rank-r adapter for a d_in x d_out matrix."""
    raise NotImplementedError("your turn")

def merge_lora(W, A, B, scale):
    """Return a new matrix W + scale * A @ B. Do not change W."""
    raise NotImplementedError("your turn")

In [10]:
assert lora_param_count(din, dout, r) == 40
assert lora_param_count(4096, 4096, 8) == 65536
merged = merge_lora(Wbase, A, B, scale)
assert np.array_equal(Wbase, frozen)                                   # the base matrix is untouched
assert np.allclose(Xtest @ merged, Xtest @ (Wbase + scale * A @ B))    # same outputs, one matrix
print("share of a 4096 x 4096 matrix at rank 8:", lora_param_count(4096, 4096, 8) / 4096**2)
print("merged test MSE:", np.mean((Xtest @ merged - Ytest) ** 2))
print("B passed")

share of a 4096 x 4096 matrix at rank 8: 0.00390625
merged test MSE: 1.3255859644546835e-09
B passed


## Discussion

1. Why is only one LoRA factor initialised to zero?
2. Should prompt tokens be removed from the input for response-only fine-tuning?
3. What must be saved with an adapter so that someone else can reproduce your outputs?
4. The toy picked its ordering on 40 development queries. What would change if you had picked it on the test queries?

## From the toy to your task

Task 12 and Task 9 teams can try three routes on the same development split: an encoder classifier, a few-shot prompt, and a LoRA adapter on a small instruction-tuned model. Record the prompt text, the demonstrations and their order beside every prompted score.

`course/extensions/hf_lora.py` trains a real LoRA adapter with response-only labels. It needs extra packages and a model download; read the [extension guide](../extensions/README.md) first.

Submit this notebook with both tests passing and a 150–250-word note: the rank you would choose for the toy and why, and one way the in-context toy differs from a real model.